# Projeto da disciplina CCM-109 - Tópicos especiais de IA - Deep Learning



In [18]:
# @title Instalar dependências

%pip install deepface opencv-python tqdm tf-keras


Note: you may need to restart the kernel to use updated packages.


In [19]:
# @title Imports

import os
import cv2
import numpy as np
from pathlib import Path
from collections import defaultdict
from tqdm.notebook import tqdm
from deepface import DeepFace

In [20]:
# @title Configurações do pipeline

VIDEO_PATH = "/dataset/FaceForensics/manipulated_sequences/DeepFakeDetection/c23/videos/01_02__exit_phone_room__YVGY8LOK.mp4"
OUTPUT_DIR = "/dataset/FaceForensics/manipulated_sequences/DeepFakeDetection/c23/frames"

DETECTOR_BACKEND = "retinaface"      
CONFIDENCE_THRESHOLD = 0.9           
ALIGN = True                          

SKIP_FRAMES = 10                    # 0 = todos; 4 = pula 4 (processa 1 a cada 5)
MAX_FRAMES = 30                     # None = todos; ex: 100 = máx 100 frames salvos
MAX_READ_LIMIT = 300                 

In [21]:
# @title Extrair rostos de um frame

def extract_faces_from_frame(frame, detector_backend="retinaface", 
                             confidence_threshold=0.85, align=True):
    """
    Extrai TODOS os rostos detectados em um frame.
    
    Retorna lista de dicts, cada um com:
        - 'face': numpy array do rosto recortado (RGB)
        - 'confidence': score de confiança da detecção
        - 'facial_area': dict com x, y, w, h da bbox no frame original
        - 'face_id': índice do rosto neste frame (0, 1, 2...)
    """
    try:
        faces = DeepFace.extract_faces(
            img_path=frame,
            detector_backend=detector_backend,
            enforce_detection=False,
            align=align
        )
    except Exception as e:
        return []
    
    results = []
    for idx, face_info in enumerate(faces):
        confidence = face_info.get("confidence", 0)
        if confidence >= confidence_threshold:
            face_img = face_info["face"]
            
            if face_img.dtype in (np.float32, np.float64):
                face_img = (face_img * 255).astype(np.uint8)
            
            results.append({
                "face": face_img,
                "confidence": confidence,
                "facial_area": face_info.get("facial_area", {}),
                "face_id": idx,  # ID do rosto dentro deste frame
            })
    
    results.sort(key=lambda x: x["confidence"], reverse=True)
    return results

In [ ]:
# @title Processar vídeo com múltiplos rostos

def process_video_multi_face(video_path, output_dir, 
                             detector="retinaface",
                             confidence=0.85,
                             align=True,
                             skip_frames=0,
                             max_frames=30,
                             max_read_limit=None):
    """
    Varre o vídeo até garantir 'max_frames' válidos contendo rostos para o dataset.
    """
    video_path = Path(video_path)
    video_name = video_path.stem
    
    frames_dir = Path(output_dir) / video_name
    frames_dir.mkdir(parents=True, exist_ok=True)
    
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise ValueError(f"Não foi possível abrir o vídeo: {video_path}")
    
    total_frames_video = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    duration = total_frames_video / fps if fps > 0 else 0
    
    print(f"🎬 Vídeo: {video_name}")
    print(f"   Total no arquivo: {total_frames_video} frames | FPS: {fps:.2f} | Duração: {duration:.1f}s")
    print(f"   Detector: {detector} | Pular frames: {skip_frames} | Máx frames salvos: {max_frames}")
    print("-" * 60)
    
    stats = {
        "video": video_name,
        "total_frames_video": total_frames_video,
        "frames_processed": 0,
        "frames_with_faces": 0,
        "frames_without_faces": 0,
        "total_faces_extracted": 0,
        "faces_per_frame": defaultdict(int),
        "saved_files": [],
    }
    
    frame_idx = 0           
    saved_frames_count = 0 
    
    pbar = tqdm(total=max_frames if max_frames else total_frames_video, desc="Processando", unit="frame")
    
    while True:
        if max_frames is not None and saved_frames_count >= max_frames:
            break
            
        if max_read_limit is not None and frame_idx >= max_read_limit:
            print(f"\n[Aviso] Trava de segurança atingida ({max_read_limit} frames lidos). Parando busca.")
            break
            
        ret, frame = cap.read()
        if not ret:
            break 
        
        if skip_frames > 0 and frame_idx % (skip_frames + 1) != 0:
            frame_idx += 1
            continue
        
        faces = extract_faces_from_frame(
            frame, 
            detector_backend=detector,
            confidence_threshold=confidence,
            align=align
        )
        
        if faces:
            stats["frames_with_faces"] += 1
            stats["faces_per_frame"][frame_idx] = len(faces)
            
            for face_idx, face_data in enumerate(faces):
                filename = f"frame_{frame_idx + 1}_face_{face_idx}.jpg"
                output_path = frames_dir / filename
                
                face_bgr = cv2.cvtColor(face_data["face"], cv2.COLOR_RGB2BGR)
                cv2.imwrite(str(output_path), face_bgr)
                
                stats["total_faces_extracted"] += 1
                stats["saved_files"].append({
                    "file": str(output_path),
                    "frame_original_idx": frame_idx,
                    "face_id": face_idx,
                    "confidence": face_data["confidence"],
                    "bbox": face_data["facial_area"],
                })
            
            saved_frames_count += 1
            pbar.update(1)
        else:
            stats["frames_without_faces"] += 1
        
        frame_idx += 1
        stats["frames_processed"] += 1
        
    pbar.close()
    cap.release()
    
    print(f"\n✅ Concluído!")
    print(f"   Frames efetivamente analisados: {stats['frames_processed']}")
    print(f"   Frames com rostos válidos extraídos: {stats['frames_with_faces']} / {max_frames}")
    print(f"   Frames descartados (sem rosto/baixa confiança): {stats['frames_without_faces']}")
    print(f"   Total de recortes de rostos salvos: {stats['total_faces_extracted']}")
    print(f"   Salvos em: {frames_dir}")
    
    return stats

In [23]:
# ============================================================
# EXECUÇÃO
# ============================================================

stats = process_video_multi_face(
    video_path=VIDEO_PATH,
    output_dir=OUTPUT_DIR,
    detector=DETECTOR_BACKEND,
    confidence=CONFIDENCE_THRESHOLD,
    align=ALIGN,
    skip_frames=SKIP_FRAMES,
    max_frames=MAX_FRAMES,
    max_read_limit=MAX_READ_LIMIT,
)

🎬 Vídeo: 01_02__exit_phone_room__YVGY8LOK
   Total no arquivo: 210 frames | FPS: 24.00 | Duração: 8.8s
   Garantir até 30 frames com rostos salvos (Skip: 10)
------------------------------------------------------------


Extraindo rostos válidos:   0%|          | 0/30 [00:00<?, ?frame_valido/s]


✅ Concluído!
   Frames efetivamente analisados: 20
   Frames com rostos válidos extraídos: 20 / 30
   Frames descartados (sem rosto/baixa confiança): 0
   Total de recortes de rostos salvos: 20
   Salvos em: /dataset/FaceForensics/manipulated_sequences/DeepFakeDetection/c23/frames/01_02__exit_phone_room__YVGY8LOK
